# Fase 0 — Post-procesado de reconexión de componentes

Evaluación, sin reentrenar, de un post-procesado quirúrgico (reconexión de componentes cercanas respetando el grosor local del vaso) sobre las predicciones ya guardadas de los tres modelos (baseline, clDice v1 y clDice v2). El objetivo es determinar en qué medida mejora la cobertura de la componente conexa principal (≈ 48 % en los tres modelos) y, por tanto, si el problema de continuidad puede abordarse sin intervenir en el entrenamiento.


In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
import os
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM_nnUNet")
NNUNET_RAW = BASE / "nnUNet_raw"
NNUNET_RESULTS = BASE / "nnUNet_results"

os.environ["nnUNet_raw"] = str(NNUNET_RAW)
os.environ["nnUNet_results"] = str(NNUNET_RESULTS)


In [3]:
!pip install -q SimpleITK


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 30.8 MB/s eta 0:00:00


In [4]:
import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy import ndimage, stats
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

pd.set_option("display.max_columns", None)


## 1. Función de reconexión quirúrgica (bridging)

Reconecta componentes conexas separadas por un hueco pequeño (`max_gap` vóxeles), limitando el grosor del puente al grosor local real del vaso en cada componente para minimizar los falsos positivos introducidos.


In [5]:
import numpy as np
from scipy import ndimage
from scipy.spatial import cKDTree


def connected_components_labeled(mask, connectivity=3):
    structure = ndimage.generate_binary_structure(mask.ndim, connectivity)
    labeled, n = ndimage.label(mask.astype(bool), structure=structure)
    return labeled, n


def crop_to_bounding_box(mask, margin=10):
    idx = np.argwhere(mask)
    if idx.size == 0:
        return mask, np.array([0, 0, 0])
    mins = np.clip(idx.min(axis=0) - margin, 0, None)
    maxs = np.minimum(idx.max(axis=0) + margin + 1, mask.shape)
    cropped = mask[mins[0]:maxs[0], mins[1]:maxs[1], mins[2]:maxs[2]]
    return cropped, mins


def paste_back(cropped, offset, full_shape):
    full = np.zeros(full_shape, dtype=cropped.dtype)
    full[offset[0]:offset[0]+cropped.shape[0],
         offset[1]:offset[1]+cropped.shape[1],
         offset[2]:offset[2]+cropped.shape[2]] = cropped
    return full


def _draw_thick_line(shape, p1, p2, radius):
    """Dibuja un segmento entre p1 y p2 y lo engorda 'radius' voxeles.
    Trabaja en una sub-caja local alrededor del segmento, no en todo el
    volumen, para que sea barato."""
    p1 = np.array(p1); p2 = np.array(p2)
    lo = np.clip(np.minimum(p1, p2) - radius - 1, 0, None)
    hi = np.minimum(np.maximum(p1, p2) + radius + 2, shape)

    local_shape = tuple(hi - lo)
    local = np.zeros(local_shape, dtype=bool)

    n_steps = int(np.linalg.norm(p2 - p1) * 2) + 2
    for t in np.linspace(0, 1, n_steps):
        point = (p1 + t * (p2 - p1) - lo).round().astype(int)
        point = np.clip(point, 0, np.array(local_shape) - 1)
        local[tuple(point)] = True

    structure = ndimage.generate_binary_structure(3, 1)
    local = ndimage.binary_dilation(local, structure=structure, iterations=radius)

    return local, lo, hi


def bridge_nearby_components(mask, max_gap=4, connectivity=3,
                              bridge_radius=2, max_iterations=200):
    """
    Version rapida basada en KDTree (sin distance_transform_edt sobre todo
    el volumen). Para cada componente construye un KDTree de sus voxeles;
    para encontrar el par mas cercano, consulta contra los KDTree de las
    demas componentes (consulta O(log N) por punto, no O(N) por voxel de
    fondo como con distance_transform_edt).

    El puente entre dos componentes es un segmento recto entre los dos
    puntos mas cercanos, engordado 'bridge_radius' voxeles -- mas simple
    que un corredor geodesico adaptativo, pero muchisimo mas rapido y
    predecible en el numero de voxeles que anade.
    """
    mask = mask.astype(bool).copy()
    added_voxels_total = 0
    iterations = 0

    while iterations < max_iterations:
        iterations += 1
        labeled, n = connected_components_labeled(mask, connectivity)
        if n <= 1:
            break

        points = [np.argwhere(labeled == k) for k in range(1, n + 1)]
        trees = [cKDTree(pts) for pts in points]

        best_gap = np.inf
        best_pair = None
        best_points = None

        for i in range(n):
            for j in range(i + 1, n):
                dists, idxs = trees[i].query(points[j])
                k = np.argmin(dists)
                gap = dists[k]
                if gap < best_gap:
                    best_gap = gap
                    best_pair = (i, j)
                    best_points = (points[i][idxs[k]], points[j][k])

        if best_gap > max_gap:
            break

        p_i, p_j = best_points
        bridge_local, lo, hi = _draw_thick_line(mask.shape, p_i, p_j, bridge_radius)

        region = mask[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]
        new_voxels = bridge_local & (~region)
        region |= bridge_local
        mask[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]] = region

        added_voxels_total += int(new_voxels.sum())

    return mask.astype(np.uint8), added_voxels_total


def bridge_nearby_components_full_volume(mask, max_gap=4, connectivity=3,
                                          bridge_radius=2, max_iterations=200,
                                          crop_margin=10):
    cropped, offset = crop_to_bounding_box(mask, margin=crop_margin)
    bridged_cropped, added = bridge_nearby_components(
        cropped, max_gap=max_gap, connectivity=connectivity,
        bridge_radius=bridge_radius, max_iterations=max_iterations,
    )
    bridged_full = paste_back(bridged_cropped, offset, mask.shape)
    return bridged_full, added


def largest_component_coverage(pred, gt, connectivity=3):
    pred = pred.astype(bool); gt = gt.astype(bool)
    labeled, n = connected_components_labeled(pred, connectivity)
    if n == 0:
        return 0.0
    sizes = ndimage.sum(pred, labeled, range(1, n + 1))
    largest = (labeled == (np.argmax(sizes) + 1))
    return np.logical_and(largest, gt).sum() / (gt.sum() + 1e-6)


def dice_score(pred, gt, eps=1e-6):
    pred = pred.astype(bool); gt = gt.astype(bool)
    inter = np.logical_and(pred, gt).sum()
    return (2.0 * inter + eps) / (pred.sum() + gt.sum() + eps)


def n_components(mask, connectivity=3):
    _, n = connected_components_labeled(mask, connectivity)
    return n


## 2. Funciones de métricas (idénticas a la evaluación anterior)


In [6]:
def load_binary_mask(path):
    img = sitk.ReadImage(str(path))
    arr = sitk.GetArrayFromImage(img)
    return (arr > 0).astype(np.uint8)


## 3. Aplicación a los tres modelos (40 casos de validación cada uno)

El bridging es más lento que las métricas anteriores, ya que recalcula las transformadas de distancia en cada iteración. Con 14-17 componentes por caso y 40 casos × 3 modelos, el cálculo requiere varios minutos.


In [7]:
DATASET_NAME  = "Dataset501_ImageCAS"
PLANS_NAME    = "nnUNetPlans"
CONFIGURATION = "3d_fullres"
FOLD          = 0
MAX_GAP       = 6  # voxeles de hueco maximo a reconectar (punto de partida; calibrar)
BRIDGE_RADIUS = 2  # grosor del puente (aprox. radio tipico de una rama fina)

TRAINERS = {
    "Dice+CE (baseline)":        "nnUNetTrainer_100epochs",
    "Dice+CE+soft-clDice v1":    "nnUNetTrainerSoftClDice100epochs",
    "Dice+CE+soft-clDice v2":    "nnUNetTrainerSoftClDiceV2_100epochs",
}

def validation_folder(trainer_name):
    return (NNUNET_RESULTS / DATASET_NAME
            / f"{trainer_name}__{PLANS_NAME}__{CONFIGURATION}"
            / f"fold_{FOLD}" / "validation")

VAL_FOLDERS = {label: validation_folder(tr) for label, tr in TRAINERS.items()}
LABELS_DIR = NNUNET_RAW / DATASET_NAME / "labelsTr"

def list_case_ids(folder):
    return sorted(f.stem.replace(".nii", "") for f in folder.glob("*.nii.gz"))

common_cases = sorted(set(list_case_ids(list(VAL_FOLDERS.values())[0])))
print(len(common_cases), "casos de validacion")


40 casos de validacion


In [8]:
results = []

for label, folder in VAL_FOLDERS.items():
    for case_id in tqdm(common_cases, desc=f"Post-procesando: {label}"):
        pred = load_binary_mask(folder / f"{case_id}.nii.gz")
        gt = load_binary_mask(LABELS_DIR / f"{case_id}.nii.gz")

        dice_before = dice_score(pred, gt)
        coverage_before = largest_component_coverage(pred, gt)
        ncomp_before = n_components(pred)

        bridged, added = bridge_nearby_components_full_volume(pred, max_gap=MAX_GAP, bridge_radius=BRIDGE_RADIUS)

        dice_after = dice_score(bridged, gt)
        coverage_after = largest_component_coverage(bridged, gt)
        ncomp_after = n_components(bridged)

        results.append({
            "model": label,
            "case_id": case_id,
            "dice_before": dice_before, "dice_after": dice_after,
            "coverage_before": coverage_before, "coverage_after": coverage_after,
            "ncomp_before": ncomp_before, "ncomp_after": ncomp_after,
            "voxeles_anadidos": added,
        })

df = pd.DataFrame(results)
df.head()


Post-procesando: Dice+CE (baseline):   0%|          | 0/40 [00:00<?, ?it/s]

Post-procesando: Dice+CE+soft-clDice v1:   0%|          | 0/40 [00:00<?, ?it/s]

Post-procesando: Dice+CE+soft-clDice v2:   0%|          | 0/40 [00:00<?, ?it/s]

,model,case_id,dice_before,dice_after,coverage_before,coverage_after,ncomp_before,ncomp_after,voxeles_anadidos
0,Dice+CE (baseline),ImageCAS_0011,0.697345,0.695952,0.451368,0.451368,17,12,347
1,Dice+CE (baseline),ImageCAS_0017,0.711042,0.710770,0.339008,0.377059,17,12,215
2,Dice+CE (baseline),ImageCAS_0018,0.709800,0.709539,0.525682,0.529337,15,12,162
3,Dice+CE (baseline),ImageCAS_0021,0.609545,0.608836,0.517066,0.517066,34,28,276
4,Dice+CE (baseline),ImageCAS_0028,0.785743,0.785743,0.390265,0.390265,5,5,0


## 4. Resumen: efecto del bridging sobre la cobertura


In [9]:
summary = df.groupby("model").agg(
    dice_before=("dice_before", "mean"),
    dice_after=("dice_after", "mean"),
    coverage_before=("coverage_before", "mean"),
    coverage_after=("coverage_after", "mean"),
    ncomp_before=("ncomp_before", "mean"),
    ncomp_after=("ncomp_after", "mean"),
    voxeles_anadidos_medio=("voxeles_anadidos", "mean"),
).round(4)

summary


,dice_before,dice_after,coverage_before,coverage_after,ncomp_before,ncomp_after,voxeles_anadidos_medio
model,,,,,,,
Dice+CE (baseline),0.7459,0.7456,0.4810,0.4865,17.150,13.700,198.3
Dice+CE+soft-clDice v1,0.7549,0.7547,0.4829,0.4852,14.725,12.075,151.7
Dice+CE+soft-clDice v2,0.7482,0.7479,0.4798,0.4814,16.100,13.550,142.2


In [10]:
for label in TRAINERS.keys():
    sub = df[df["model"] == label]
    w_stat, w_p = stats.wilcoxon(sub["coverage_before"], sub["coverage_after"])
    print(f"{label}: Wilcoxon cobertura antes vs despues -> statistic={w_stat:.2f}, p={w_p:.4f}")


Dice+CE (baseline): Wilcoxon cobertura antes vs despues -> statistic=0.00, p=0.0000
Dice+CE+soft-clDice v1: Wilcoxon cobertura antes vs despues -> statistic=14.00, p=0.0031
Dice+CE+soft-clDice v2: Wilcoxon cobertura antes vs despues -> statistic=9.00, p=0.0186


## 5. Guardar resultados

In [11]:
output_dir = BASE / "evaluacion_comparativa"
output_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(output_dir / "resultados_postprocesado_bridging.csv", index=False)
summary.to_csv(output_dir / "resumen_postprocesado_bridging.csv")

print("Guardado en:", output_dir)


Guardado en: /content/drive/MyDrive/TFM_nnUNet/evaluacion_comparativa
